# Proyecto BI - Uber NYC (2009-2015)
## Análisis de KPIs - Justificación de Negocio e Interpretación

Este notebook define, justifica e interpreta 7 KPIs clave para el análisis de negocio de Uber en Nueva York durante el periodo 2009-2015.

### Contexto del Negocio

**Hipótesis Fundamentales del Negocio:**
1. Uber está en fase de expansión acelerada (periodo 2009-2015)
2. La demanda muestra patrones estacionales y temporales predecibles
3. La optimización de la flota depende de patrones horarios
4. La rentabilidad varía según segmentos de distancia y ocupación
5. La retención de usuarios y fidelización impacta directamente en ingresos sostenibles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Bibliotecas importadas correctamente")

## Carga de Datos

Cargamos el dataset limpio resultante del proceso ETL.

In [ ]:
df = pd.read_csv('uber_clean.csv')

print("=== DATASET UBER NYC (2009-2015) ===")
print(f"Total de registros: {len(df):,}")
print(f"Rango temporal: {df['year'].min()} - {df['year'].max()}")
print(f"\nColumnas disponibles: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

---

## KPI 1: Tasa de Crecimiento de Ingresos (Revenue Growth Rate)

### **Definición del KPI:**
```
Tasa de Crecimiento = ((Ingresos_Periodo_Actual - Ingresos_Periodo_Anterior) / Ingresos_Periodo_Anterior) × 100
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "Uber está en fase de expansión acelerada"

Este KPI es **crítico** porque:
1. **Valida el modelo de negocio:** Mide si la estrategia de expansión está generando retornos
2. **Atracción de inversionistas:** Demuestra potencial de crecimiento a stakeholders
3. **Asignación de recursos:** Justifica inversión en marketing, expansión geográfica y tecnología
4. **Benchmarking competitivo:** Permite comparar con alternativas de transporte tradicional
5. **Planificación estratégica:** Ayuda a predecir necesidades de infraestructura y personal

### **Interpretación del Indicador:**
- **Crecimiento > 20%:** Expansión agresiva exitosa, invertir en escala
- **Crecimiento 10-20%:** Crecimiento saludable, mantener estrategia
- **Crecimiento 0-10%:** Maduración del mercado, buscar nuevas oportunidades
- **Crecimiento negativo:** Alerta temprana de problemas, requiere intervención inmediata

In [ ]:
print("=" * 60)
print("KPI 1: TASA DE CRECIMIENTO DE INGRESOS")
print("=" * 60)

# Calcular ingresos por año
ingresos_por_anio = df.groupby('year')['fare_amount'].agg(['sum', 'count'])
ingresos_por_anio.columns = ['total_ingresos', 'total_viajes']

# Calcular tasa de crecimiento año sobre año
ingresos_por_anio['tasa_crecimiento'] = ingresos_por_anio['total_ingresos'].pct_change() * 100
ingresos_por_anio['crecimiento_viajes'] = ingresos_por_anio['total_viajes'].pct_change() * 100

print("\n### INGRESOS POR AÑO Y TASA DE CRECIMIENTO ###")
print(ingresos_por_anio.to_string())

print("\n### INTERPRETACIÓN ###")
print(f"Tasa de crecimiento promedio: {ingresos_por_anio['tasa_crecimiento'].mean():.2f}%")
print(f"Año de máximo crecimiento: {ingresos_por_anio['tasa_crecimiento'].idxmax()}")
print(f"Máxima tasa de crecimiento: {ingresos_por_anio['tasa_crecimiento'].max():.2f}%")

In [ ]:
# Visualización: Crecimiento de Ingresos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Ingresos totales por año
axes[0].plot(ingresos_por_anio.index, ingresos_por_anio['total_ingresos'], 
             marker='o', linewidth=2, markersize=8, color='darkblue')
axes[0].set_title('Ingresos Totales por Año', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Ingresos (USD)')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Gráfico 2: Tasa de crecimiento
colors = ['green' if x > 0 else 'red' for x in ingresos_por_anio['tasa_crecimiento'].dropna()]
axes[1].bar(ingresos_por_anio.index[1:], ingresos_por_anio['tasa_crecimiento'].dropna(), 
           color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[1].set_title('Tasa de Crecimiento Año sobre Año', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Tasa de Crecimiento (%)')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n### CONCLUSIÓN DE NEGOCIO ###")
if ingresos_por_anio['tasa_crecimiento'].mean() > 20:
    print("✅ FASE DE EXPANSIÓN AGRESIVA: Estrategia validada, continuar inversión en escala")
elif ingresos_por_anio['tasa_crecimiento'].mean() > 10:
    print("✅ CRECIMIENTO SALUDABLE: Mantener estrategia actual y optimizar operaciones")
else:
    print("⚠️ MADURACIÓN DEL MERCADO: Buscar nuevas oportunidades de crecimiento")

---

## KPI 2: Ticket Promedio (Average Fare Amount)

### **Definición del KPI:**
```
Ticket Promedio = Total Ingresos / Total Viajes
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La rentabilidad varía según segmentos de distancia y ocupación"

Este KPI es **fundamental** porque:
1. **Poder de pricing:** Mide el valor promedio que los clientes están dispuestos a pagar
2. **Segmentación de mercado:** Identifica viajes de alto valor vs. viajes de comodidad
3. **Optimización de tarifas:** Permite ajustar pricing dinámico por zona/horario
4. **Proyección de ingresos:** Base para modelar escenarios de crecimiento
5. **Comparativa competitiva:** Benchmark contra taxis tradicionales y otros rideshare

### **Interpretación del Indicador:**
- **Ticket > $15:** Segmento premium, mayor margen por viaje
- **Ticket $10-15:** Rango óptimo de equilibrio volumen-valor
- **Ticket $5-10:** Segmento alta frecuencia, bajo margen
- **Ticket < $5:** Alerta de posible subsidización excesiva

In [ ]:
print("=" * 60)
print("KPI 2: TICKET PROMEDIO")
print("=" * 60)

# Calcular ticket promedio general
ticket_promedio_global = df['fare_amount'].sum() / len(df)

# Ticket promedio por año
ticket_por_anio = df.groupby('year').agg({
    'fare_amount': ['sum', 'count']
}).reset_index()
ticket_por_anio.columns = ['year', 'total_ingresos', 'total_viajes']
ticket_por_anio['ticket_promedio'] = ticket_por_anio['total_ingresos'] / ticket_por_anio['total_viajes']

# Ticket promedio por franja horaria
ticket_por_franja = df.groupby('time_of_day').agg({
    'fare_amount': ['sum', 'count']
 }).reset_index()
ticket_por_franja.columns = ['franja', 'total_ingresos', 'total_viajes']
ticket_por_franja['ticket_promedio'] = ticket_por_franja['total_ingresos'] / ticket_por_franja['total_viajes']
ticket_por_franja = ticket_por_franja.sort_values('ticket_promedio', ascending=False)

print("\n### TICKET PROMEDIO GLOBAL ###")
print(f"${ticket_promedio_global:.2f} por viaje")

print("\n### TICKET PROMEDIO POR AÑO ###")
print(ticket_por_anio[['year', 'ticket_promedio']].to_string(index=False))

print("\n### TICKET PROMEDIO POR FRANJA HORARIA ###")
print(ticket_por_franja[['franja', 'ticket_promedio', 'total_viajes']].to_string(index=False))

In [ ]:
# Visualización: Ticket Promedio
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Ticket promedio por año
axes[0].plot(ticket_por_anio['year'], ticket_por_anio['ticket_promedio'], 
             marker='s', linewidth=2, markersize=8, color='purple')
axes[0].axhline(y=ticket_promedio_global, color='red', linestyle='--', 
               label=f'Promedio Global: ${ticket_promedio_global:.2f}')
axes[0].set_title('Ticket Promedio por Año', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Ticket Promedio (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Gráfico 2: Ticket promedio por franja horaria
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = axes[1].bar(ticket_por_franja['franja'], ticket_por_franja['ticket_promedio'], 
                  color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(y=ticket_promedio_global, color='red', linestyle='--', 
               label=f'Promedio: ${ticket_promedio_global:.2f}')
axes[1].set_title('Ticket Promedio por Franja Horaria', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Franja Horaria')
axes[1].set_ylabel('Ticket Promedio (USD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Añadir etiquetas de valor
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'${height:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n### INSIGHTS DE NEGOCIO ###")
print(f"Segmento de mayor valor: {ticket_por_franja.iloc[0]['franja']} (${ticket_por_franja.iloc[0]['ticket_promedio']:.2f})")
print(f"Segmento de menor valor: {ticket_por_franja.iloc[-1]['franja']} (${ticket_por_franja.iloc[-1]['ticket_promedio']:.2f})")
print(f"\nDiferencial valor alto-bajo: ${(ticket_por_franja.iloc[0]['ticket_promedio'] - ticket_por_franja.iloc[-1]['ticket_promedio']):.2f} ({(ticket_por_franja.iloc[0]['ticket_promedio'] / ticket_por_franja.iloc[-1]['ticket_promedio'] - 1) * 100:.1f}% más alto)")

---

## KPI 3: Factor de Ocupación (Passenger Utilization Rate)

### **Definición del KPI:**
```
Factor de Ocupación = Pasajeros Promedio por Viaje / Capacidad Máxima (6 pasajeros)
Pasajeros Promedio = Total Pasajeros Transportados / Total Viajes
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La retención de usuarios y optimización de flota impactan directamente en ingresos"

Este KPI es **crítico** porque:
1. **Eficiencia de la flota:** Muestra qué tan bien se está utilizando la capacidad disponible
2. **Impacto ambiental:** Factor ESG clave (más pasajeros por viaje = menor huella de carbono)
3. **Maximización de ingresos:** Cada pasajero adicional es ingreso puro sin costo marginal significativo
4. **Optimización de rutas:** Patrones de ocupación altos pueden indicar rutas consolidadas
5. **Marketing estratégico:** Permite diseñar campañas para fomentar viajes compartidos

### **Interpretación del Indicador:**
- **Ocupación > 50%:** Excelente uso de capacidad, alto impacto ambiental positivo
- **Ocupación 30-50%:** Uso moderado, oportunidad para promover ridesharing
- **Ocupación < 30%:** Alta dependencia de viajes individuales, margen de mejora

In [ ]:
print("=" * 60)
print("KPI 3: FACTOR DE OCUPACIÓN")
print("=" * 60)

# Calcular estadísticas de ocupación
total_pasajeros = df['passenger_count'].sum()
total_viajes = len(df)
pasajeros_promedio = total_pasajeros / total_viajes
capacidad_maxima = 6
factor_ocupacion = pasajeros_promedio / capacidad_maxima

# Distribución de viajes por número de pasajeros
distribucion_pasajeros = df['passenger_count'].value_counts().sort_index()
distribucion_pasajeros_pct = (distribucion_pasajeros / total_viajes * 100).round(2)

# Ocupación por franja horaria
ocupacion_por_franja = df.groupby('time_of_day')['passenger_count'].agg(['mean', 'count'])
ocupacion_por_franja.columns = ['pasajeros_promedio', 'total_viajes']
ocupacion_por_franja['factor_ocupacion'] = (ocupacion_por_franja['pasajeros_promedio'] / capacidad_maxima * 100).round(2)
ocupacion_por_franja = ocupacion_por_franja.sort_values('factor_ocupacion', ascending=False)

print("\n### ESTADÍSTICAS GENERALES DE OCUPACIÓN ###")
print(f"Total pasajeros transportados: {total_pasajeros:,}")
print(f"Pasajeros promedio por viaje: {pasajeros_promedio:.2f}")
print(f"Factor de ocupación: {factor_ocupacion:.2%}")

print("\n### DISTRIBUCIÓN DE VIAJES POR NÚMERO DE PASAJEROS ###")
for pasajeros, conteo in distribucion_pasajeros.items():
    porcentaje = distribucion_pasajeros_pct[pasajeros]
    print(f"{pasajeros} pasajero(s): {conteo:,} viajes ({porcentaje:.2f}%)")

print("\n### FACTOR DE OCUPACIÓN POR FRANJA HORARIA ###")
print(ocupacion_por_franja.to_string())

In [ ]:
# Visualización: Factor de Ocupación
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Distribución de viajes por número de pasajeros
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']
bars = axes[0].bar(distribucion_pasajeros.index, distribucion_pasajeros.values, 
                   color=colors[:len(distribucion_pasajeros)], alpha=0.7, edgecolor='black')
axes[0].set_title('Distribución de Viajes por Número de Pasajeros', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de Pasajeros')
axes[0].set_ylabel('Cantidad de Viajes')
axes[0].grid(True, alpha=0.3, axis='y')

# Añadir etiquetas de porcentaje
for i, (bar, pct) in enumerate(zip(bars, distribucion_pasajeros_pct)):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

# Gráfico 2: Factor de ocupación por franja horaria
orden_franjas = ocupacion_por_franja.index
valores_ocupacion = ocupacion_por_franja['factor_ocupacion'].values
colors2 = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

bars2 = axes[1].bar(range(len(orden_franjas)), valores_ocupacion, 
                    color=colors2[:len(orden_franjas)], alpha=0.7, edgecolor='black')
axes[1].axhline(y=factor_ocupacion * 100, color='red', linestyle='--', linewidth=2,
               label=f'Promedio Global: {factor_ocupacion:.1%}')
axes[1].set_title('Factor de Ocupación por Franja Horaria', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Franja Horaria')
axes[1].set_ylabel('Factor de Ocupación (%)')
axes[1].set_xticks(range(len(orden_franjas)))
axes[1].set_xticklabels(orden_franjas)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Añadir etiquetas de valor
for bar, valor in zip(bars2, valores_ocupacion):
    axes[1].text(bar.get_x() + bar.get_width()/2., valor,
                f'{valor:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n### CONCLUSIÓN DE NEGOCIO ###")
if factor_ocupacion > 0.5:
    print("✅ EXCELENTE EFICIENCIA: Uso óptimo de capacidad, alto impacto ESG positivo")
elif factor_ocupacion > 0.3:
    print("✅ EFICIENCIA MODERADA: Oportunidad para promocionar viajes compartidos")
else:
    print("⚠️ BAJA EFICIENCIA: Alta dependencia de viajes individuales, margen de mejora significativo")

---

## KPI 4: Índice de Estacionalidad de la Demanda (Demand Seasonality Index)

### **Definición del KPI:**
```
Índice de Estacionalidad = (Viajes en Período / Viajes Promedio Mensual) × 100
Coeficiente de Variación Mensual = (Desviación Estándar Viajes Mensuales / Media Viajes Mensuales)
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La demanda muestra patrones estacionales y temporales predecibles"

Este KPI es **estratégico** porque:
1. **Planificación de capacidad:** Permite anticipar picos y valles de demanda
2. **Gestión de inventario:** Ajustar número de conductores según temporada
3. **Campañas marketing:** Timing óptimo para promociones y adquisición de usuarios
4. **Forecasting financiero:** Predicción más precisa de ingresos futuros
5. **Gestión de costos:** Optimizar gastos operativos según demanda proyectada

### **Interpretación del Indicador:**
- **CV > 0.3:** Alta estacionalidad, requiere planificación detallada por temporada
- **CV 0.2-0.3:** Estacionalidad moderada, planificación por trimestres
- **CV < 0.2:** Baja estacionalidad, demanda relativamente estable

In [ ]:
print("=" * 60)
print("KPI 4: ÍNDICE DE ESTACIONALIDAD DE LA DEMANDA")
print("=" * 60)

# Viajes por mes (agregando todos los años)
viajes_por_mes = df.groupby('month').size()
viajes_por_mes.name = 'total_viajes'

# Estadísticas de estacionalidad
promedio_mensual = viajes_por_mes.mean()
desviacion_estandar = viajes_por_mes.std()
coeficienteVariacion = desviacion_estandar / promedio_mensual

# Crear índice de estacionalidad
indice_estacionalidad = (viajes_por_mes / promedio_mensual * 100).round(2)
df_estacionalidad = pd.DataFrame({
    'mes': viajes_por_mes.index,
    'total_viajes': viajes_por_mes.values,
    'indice_estacionalidad': indice_estacionalidad.values,
    'desviacion_porcentaje': ((viajes_por_mes.values / promedio_mensual - 1) * 100).round(2)
})

# Nombres de meses
nombres_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
                 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
df_estacionalidad['nombre_mes'] = df_estacionalidad['mes'].apply(lambda x: nombres_meses[x-1])

print("\n### ESTADÍSTICAS DE ESTACIONALIDAD ###")
print(f"Promedio mensual de viajes: {promedio_mensual:,.0f}")
print(f"Desviación estándar: {desviacion_estandar:,.0f}")
print(f"Coeficiente de variación: {coeficienteVariacion:.3f}")

print("\n### ÍNDICE DE ESTACIONALIDAD POR MES ###")
print(df_estacionalidad[['nombre_mes', 'total_viajes', 'indice_estacionalidad', 'desviacion_porcentaje']].to_string(index=False))

In [ ]:
# Visualización: Índice de Estacionalidad
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Viajes por mes
axes[0].plot(df_estacionalidad['nombre_mes'], df_estacionalidad['total_viajes'], 
             marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[0].axhline(y=promedio_mensual, color='red', linestyle='--', 
               label=f'Promedio: {promedio_mensual:,.0f}')
axes[0].set_title('Viajes Totales por Mes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Cantidad de Viajes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Gráfico 2: Índice de estacionalidad
colors = ['green' if x >= 100 else 'orange' for x in df_estacionalidad['indice_estacionalidad']]
bars = axes[1].bar(df_estacionalidad['nombre_mes'], df_estacionalidad['indice_estacionalidad'], 
                   color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(y=100, color='blue', linestyle='-', linewidth=1, label='Promedio (100)')
axes[1].set_title('Índice de Estacionalidad', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Índice (Promedio = 100)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)

# Añadir etiquetas
for bar, valor in zip(bars, df_estacionalidad['indice_estacionalidad']):
    axes[1].text(bar.get_x() + bar.get_width()/2., valor,
                f'{valor:.0f}', ha='center', va='bottom' if valor >= 100 else 'top', fontsize=9)

plt.tight_layout()
plt.show()

print("\n### INSIGHTS DE NEGOCIO ###")
mes_pico = df_estacionalidad.loc[df_estacionalidad['indice_estacionalidad'].idxmax()]
mes_valle = df_estacionalidad.loc[df_estacionalidad['indice_estacionalidad'].idxmin()]

print(f"Mes de mayor demanda: {mes_pico['nombre_mes']} (índice {mes_pico['indice_estacionalidad']:.0f}, {mes_pico['desviacion_porcentaje']:+.1f}% vs promedio)")
print(f"Mes de menor demanda: {mes_valle['nombre_mes']} (índice {mes_valle['indice_estacionalidad']:.0f}, {mes_valle['desviacion_porcentaje']:+.1f}% vs promedio)")
print(f"\nVariación pico-valle: {mes_pico['desviacion_porcentaje'] - mes_valle['desviacion_porcentaje']:.1f} puntos porcentuales")

if coeficienteVariacion > 0.3:
    print("\n⚠️ ALTA ESTACIONALIDAD: Requiere planificación detallada por temporada")
elif coeficienteVariacion > 0.2:
    print("\n✅ ESTACIONALIDAD MODERADA: Planificación por trimestres es suficiente")
else:
    print("\n✅ BAJA ESTACIONALIDAD: Demanda relativamente estable durante el año")

---

## KPI 5: Eficiencia por Hora (Revenue per Hour by Time Slot)

### **Definición del KPI:**
```
Ingresos por Hora = Total Ingresos en Franja Horaria / Total Horas Operativas en Franja
Eficiencia Relativa = Ingresos por Hora de Franja / Ingresos por Hora Promedio
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La optimización de la flota depende de patrones horarios"

Este KPI es **operacionalmente crítico** porque:
1. **Asignación de conductores:** Permite optimizar turnos y horarios de mayor rentabilidad
2. **Incentivos estratégicos:** Diseñar bonos por hora en periodos de alta demanda
3. **Pricing dinámico:** Implementar surge pricing en horas de alta presión sobre la oferta
4. **Marketing timing:** Pautar publicidad en horas de menor eficiencia para atraer demanda
5. **Balance oferta-demanda:** Identificar brechas entre disponibilidad de conductores y demanda

### **Interpretación del Indicador:**
- **Eficiencia > 120%:** Hora pico, priorizar disponibilidad de conductores
- **Eficiencia 80-120%:** Hora normal, operación estándar
- **Eficiencia < 80%:** Hora valle, oportunidad para promociones

In [ ]:
print("=" * 60)
print("KPI 5: EFICIENCIA POR HORA (REVENUE PER HOUR)")
print("=" * 60)

# Ingresos por hora del día (0-23)
ingresos_por_hora = df.groupby('hour').agg({
    'fare_amount': ['sum', 'count']
}).reset_index()
ingresos_por_hora.columns = ['hour', 'total_ingresos', 'total_viajes']

# Calcular ingresos promedio por hora
ingresos_promedio_por_hora = ingresos_por_hora['total_ingresos'].mean()
ingresos_por_hora['eficiencia_porcentaje'] = (ingresos_por_hora['total_ingresos'] / ingresos_promedio_por_hora * 100).round(2)

# Calcular ticket promedio por hora
ingresos_por_hora['ticket_promedio'] = (ingresos_por_hora['total_ingresos'] / ingresos_por_hora['total_viajes']).round(2)

# Clasificar horas
def clasificar_hora(efficiencia):
    if efficiencia >= 120:
        return 'Pico'
    elif efficiencia >= 80:
        return 'Normal'
    else:
        return 'Valle'

ingresos_por_hora['clasificacion'] = ingresos_por_hora['eficiencia_porcentaje'].apply(clasificar_hora)

# Resumen por clasificación
resumen_clasificacion = ingresos_por_hora.groupby('clasificacion').agg({
    'hour': 'count',
    'total_ingresos': 'sum',
    'eficiencia_porcentaje': 'mean'
}).reset_index()
resumen_clasificacion.columns = ['clasificacion', 'num_horas', 'ingresos_totales', 'eficiencia_promedio']
resumen_clasificacion = resumen_clasificacion.sort_values('eficiencia_promedio', ascending=False)

print("\n### INGRESOS PROMEDIO POR HORA: ${:,.0f} ###".format(ingresos_promedio_por_hora))

print("\n### RESUMEN POR CLASIFICACIÓN DE EFICIENCIA ###")
print(resumen_clasificacion.to_string(index=False))

print("\n### TOP 5 HORAS DE MAYOR EFICIENCIA ###")
top5 = ingresos_por_hora.nlargest(5, 'total_ingresos')[['hour', 'total_ingresos', 'eficiencia_porcentaje', 'ticket_promedio']]
print(top5.to_string(index=False))

print("\n### BOTTOM 5 HORAS DE MENOR EFICIENCIA ###")
bottom5 = ingresos_por_hora.nsmallest(5, 'total_ingresos')[['hour', 'total_ingresos', 'eficiencia_porcentaje', 'ticket_promedio']]
print(bottom5.to_string(index=False))

In [ ]:
# Visualización: Eficiencia por Hora
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Gráfico 1: Ingresos por hora
colors = ['green' if x >= 120 else 'orange' if x >= 80 else 'red' 
          for x in ingresos_por_hora['eficiencia_porcentaje']]

bars = axes[0].bar(ingresos_por_hora['hour'], ingresos_por_hora['total_ingresos'], 
                   color=colors, alpha=0.7, edgecolor='black')
axes[0].axhline(y=ingresos_promedio_por_hora, color='blue', linestyle='--', linewidth=2,
               label=f'Promedio: ${ingresos_promedio_por_hora:,.0f}')
axes[0].set_title('Ingresos Totales por Hora del Día', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hora del Día')
axes[0].set_ylabel('Ingresos Totales (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_xticks(range(0, 24, 2))

# Gráfico 2: Eficiencia relativa por hora
axes[1].plot(ingresos_por_hora['hour'], ingresos_por_hora['eficiencia_porcentaje'], 
             marker='o', linewidth=2, markersize=6, color='darkblue')
axes[1].axhline(y=100, color='black', linestyle='-', linewidth=1, label='Promedio (100%)')
axes[1].axhline(y=120, color='green', linestyle='--', linewidth=1, label='Umbral Pico (120%)')
axes[1].axhline(y=80, color='red', linestyle='--', linewidth=1, label='Umbral Valle (80%)')
axes[1].set_title('Eficiencia Relativa por Hora', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hora del Día')
axes[1].set_ylabel('Eficiencia (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_ylim([0, max(ingresos_por_hora['eficiencia_porcentaje']) * 1.1])

# Añadir etiquetas de horas pico
horas_pico = ingresos_por_hora[ingresos_por_hora['eficiencia_porcentaje'] >= 120]
for _, row in horas_pico.iterrows():
    axes[1].annotate(f"{row['hour']}:00", 
                    xy=(row['hour'], row['eficiencia_porcentaje']),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=8, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n### INSIGHTS DE NEGOCIO ###")
horas_pico_lista = ingresos_por_hora[ingresos_por_hora['clasificacion'] == 'Pico']['hour'].tolist()
horas_valle_lista = ingresos_por_hora[ingresos_por_hora['clasificacion'] == 'Valle']['hour'].tolist()

print(f"Horas pico (priorizar disponibilidad): {horas_pico_lista}")
print(f"Horas valle (oportunidad promociones): {horas_valle_lista}")

if len(horas_pico_lista) <= 4:
    print("\n✅ CONCENTRACIÓN EFICIENTE: Picos bien definidos, fácil optimización de turnos")
else:
    print("\n⚠️ DEMANDA DISPERSA: Múltiples picos, requiere programación compleja de conductores")

---

## KPI 6: Retorno por Kilómetro (Revenue per Kilometer)

### **Definición del KPI:**
```
Retorno por KM = Total Ingresos / Total Kilómetros Recorridos
Retorno por KM por Segmento = Ingresos Segmento / Kilómetros Segmento
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La rentabilidad varía según segmentos de distancia"

Este KPI es **financieramente estratégico** porque:
1. **Evaluación de rentabilidad:** Mide la eficiencia económica de cada kilómetro operado
2. **Segmentación de mercado:** Identifica viajes cortos vs. largos y su rentabilidad diferencial
3. **Optimización de rutas:** Permite diseñar estrategias para maximizar retorno por distancia
4. **Comparativa operativa:** Benchmark contra costos operativos (combustible, mantenimiento)
5. **Diseño de incentivos:** Estructurar bonificaciones a conductores por kilómetro rentable

### **Interpretación del Indicador:**
- **Retorno > $5/KM:** Segmento premium de alta rentabilidad
- **Retorno $3-5/KM:** Rango óptimo de equilibrio
- **Retorno < $3/KM:** Segmento de baja rentabilidad, revisar estructura de costos

In [ ]:
print("=" * 60)
print("KPI 6: RETORNO POR KILÓMETRO")
print("=" * 60)

# Calcular retorno por KM global
total_ingresos = df['fare_amount'].sum()
total_kilometros = df['distance_km'].sum()
retorno_por_km_global = total_ingresos / total_kilometros

# Crear segmentos de distancia
def segmentar_distancia(km):
    if km <= 2:
        return 'Corto (≤2 km)'
    elif km <= 5:
        return 'Medio (2-5 km)'
    elif km <= 10:
        return 'Largo (5-10 km)'
    else:
        return 'Muy Largo (>10 km)'

df['segmento_distancia'] = df['distance_km'].apply(segmentar_distancia)

# Análisis por segmento
analisis_por_segmento = df.groupby('segmento_distancia').agg({
    'fare_amount': ['sum', 'mean', 'count'],
    'distance_km': ['sum', 'mean']
}).reset_index()

analisis_por_segmento.columns = ['segmento', 'ingresos_totales', 'ticket_promedio', 
                                  'num_viajes', 'kilometros_totales', 'distancia_promedio']

# Calcular retorno por KM por segmento
analisis_por_segmento['retorno_por_km'] = (analisis_por_segmento['ingresos_totales'] / 
                                           analisis_por_segmento['kilometros_totales']).round(2)

# Calcular porcentaje de ingresos y kilómetros
analisis_por_segmento['pct_ingresos'] = (analisis_por_segmento['ingresos_totales'] / 
                                          total_ingresos * 100).round(2)
analisis_por_segmento['pct_kilometros'] = (analisis_por_segmento['kilometros_totales'] / 
                                            total_kilometros * 100).round(2)

# Ordenar por retorno por KM
analisis_por_segmento = analisis_por_segmento.sort_values('retorno_por_km', ascending=False)

print("\n### RETORNO POR KM GLOBAL ###")
print(f"${retorno_por_km_global:.2f} por kilómetro")

print("\n### ANÁLISIS POR SEGMENTO DE DISTANCIA ###")
print(analisis_por_segmento.to_string(index=False))

# Identificar segmentos más y menos rentables
segmento_mas_rentable = analisis_por_segmento.iloc[0]
segmento_menos_rentable = analisis_por_segmento.iloc[-1]

print("\n### INSIGHTS DE RENTABILIDAD ###")
print(f"Segmento más rentable: {segmento_mas_rentable['segmento']} (${segmento_mas_rentable['retorno_por_km']}/km)")
print(f"Segmento menos rentable: {segmento_menos_rentable['segmento']} (${segmento_menos_rentable['retorno_por_km']}/km)")
print(f"Diferencial de rentabilidad: ${segmento_mas_rentable['retorno_por_km'] - segmento_menos_rentable['retorno_por_km']:.2f}/km ({(segmento_mas_rentable['retorno_por_km'] / segmento_menos_rentable['retorno_por_km'] - 1) * 100:.1f}% más alto)")

In [ ]:
# Visualización: Retorno por Kilómetro
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Retorno por KM por segmento
colors = ['#2ECC71', '#3498DB', '#9B59B6', '#E74C3C']
bars = axes[0].bar(analisis_por_segmento['segmento'], analisis_por_segmento['retorno_por_km'], 
                   color=colors, alpha=0.7, edgecolor='black')
axes[0].axhline(y=retorno_por_km_global, color='red', linestyle='--', linewidth=2,
               label=f'Global: ${retorno_por_km_global:.2f}/km')
axes[0].set_title('Retorno por Kilómetro por Segmento', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Segmento de Distancia')
axes[0].set_ylabel('Retorno por KM (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=15)

# Añadir etiquetas
for bar, valor in zip(bars, analisis_por_segmento['retorno_por_km']):
    axes[0].text(bar.get_x() + bar.get_width()/2., valor,
                f'${valor:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Gráfico 2: Balance ingresos vs kilómetros por segmento
x = np.arange(len(analisis_por_segmento))
width = 0.35

bars1 = axes[1].bar(x - width/2, analisis_por_segmento['pct_ingresos'], 
                     width, label='% Ingresos', color='green', alpha=0.7)
bars2 = axes[1].bar(x + width/2, analisis_por_segmento['pct_kilometros'], 
                     width, label='% Kilómetros', color='blue', alpha=0.7)

axes[1].set_title('Distribución de Ingresos vs Kilómetros', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Segmento de Distancia')
axes[1].set_ylabel('Porcentaje del Total (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(analisis_por_segmento['segmento'], rotation=15, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Añadir etiquetas de diferencia
for i, (ing, km) in enumerate(zip(analisis_por_segmento['pct_ingresos'], 
                                  analisis_por_segmento['pct_kilometros'])):
    diff = ing - km
    color = 'green' if diff > 0 else 'red'
    axes[1].annotate(f'{diff:+.1f}%', 
                    xy=(i, max(ing, km)),
                    xytext=(0, 10), textcoords='offset points',
                    fontsize=8, color=color, fontweight='bold', ha='center')

plt.tight_layout()
plt.show()

print("\n### CONCLUSIÓN DE NEGOCIO ###")
if retorno_por_km_global > 5:
    print("✅ ALTA RENTABILIDAD: Modelo de negocio muy eficiente por kilómetro")
elif retorno_por_km_global > 3:
    print("✅ RENTABILIDAD ÓPTIMA: Equilibrio saludable entre volumen y margen")
else:
    print("⚠️ RENTABILIDAD BAJO: Revisar estructura de costos y pricing")

---

## KPI 7: Índice de Fidelización de Usuarios (Repeat Usage Rate)

### **Definición del KPI:**
```
Índice de Fidelización = (Usuarios con Múltiples Viajes / Total Usuarios Únicos) × 100
Frecuencia Promedio = Total Viajes / Total Usuarios Únicos
```

### **Justificación desde el Negocio:**

**Hipótesis Relacionada:** "La retención de usuarios impacta directamente en ingresos sostenibles"

Este KPI es **estratégico para crecimiento sostenible** porque:
1. **Costo de adquisición:** Retener usuarios es 5x más económico que adquirir nuevos
2. **Valor lifetime:** Usuarios recurrentes generan mayor valor a lo largo del tiempo
3. **Ventaja competitiva:** Alta fidelización crea barreras de entrada para competidores
4. **Predicción de ingresos:** Base de usuarios leales permite forecast más preciso
5. **Marketing viral:** Usuarios fieles recomiendan el servicio, reduciendo CAC

### **Interpretación del Indicador:**
- **Fidelización > 60%:** Base de usuarios muy leales, activación de programas de referidos
- **Fidelización 40-60%:** Saludable, foco en aumentar frecuencia de uso
- **Fidelización < 40%:** Alta dependencia de adquisición, revisar experiencia de usuario

In [ ]:
print("=" * 60)
print("KPI 7: ÍNDICE DE FIDELIZACIÓN DE USUARIOS")
print("=" * 60)

# NOTA: Este dataset NO tiene identificador de usuario único,
# por lo que usamos aproximaciones basadas en patrones de uso

# Aproximación 1: Usuarios únicos estimados por día-hora-zona
# (assumiendo que mismo usuario no hace múltiples viajes simultáneos)
df['fecha'] = pd.to_datetime(df[['year', 'month']].assign(day=1))  # Aproximación

# Análisis por periodicidad (días de la semana)
viajes_por_dia = df.groupby('day_of_week').agg({
    'fare_amount': ['sum', 'count'],
}).reset_index()
viajes_por_dia.columns = ['dia_semana', 'ingresos_totales', 'total_viajes']

# Nombres de días
nombres_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
viajes_por_dia['nombre_dia'] = viajes_por_dia['dia_semana'].apply(lambda x: nombres_dias[x])

# Análisis de consistencia (proxy de fidelización)
viajes_promedio_dia = viajes_por_dia['total_viajes'].mean()
desviacion_estandar_dia = viajes_por_dia['total_viajes'].std()
coeficienteVariacion_dia = desviacion_estandar_dia / viajes_promedio_dia

# Índice de uso consistente (proxy de usuarios recurrentes)
# Si la variación entre días es baja, indica base de usuarios más estable
indice_uso_consistente = (1 - coeficienteVariacion_dia) * 100

# Distribución de viajes por hora (para detectar patrones de uso recurrente)
viajes_por_hora_fidelizacion = df.groupby('hour').agg({
    'fare_amount': 'count'
}).reset_index()
viajes_por_hora_fidelizacion.columns = ['hour', 'total_viajes']

# Identificar horas con uso consistente (indicador de usuarios habituales)
horas_con_uso_significativo = (viajes_por_hora_fidelizacion['total_viajes'] > 
                               viajes_por_hora_fidelizacion['total_viajes'].mean()).sum()
indice_cobertura_horaria = (horas_con_uso_significativo / 24 * 100)

print("\n### MÉTRICAS DE FIDELIZACIÓN (PROXIES) ###")
print(f"Índice de uso consistente por día: {indice_uso_consistente:.1f} %")
print(f"Coeficiente de variación diaria: {coeficienteVariacion_dia:.3f}")
print(f"Índice de cobertura horaria: {indice_cobertura_horaria:.1f}%")
print(f"(Horas con uso significativo: {horas_con_uso_significativo}/24)")

print("\n### DISTRIBUCIÓN DE VIAJES POR DÍA DE LA SEMANA ###")
viajes_por_dia_display = viajes_por_dia[['nombre_dia', 'total_viajes', 'ingresos_totales']].copy()
viajes_por_dia_display['pct_del_total'] = (viajes_por_dia_display['total_viajes'] / 
                                           viajes_por_dia_display['total_viajes'].sum() * 100).round(1)
print(viajes_por_dia_display.to_string(index=False))

# Clasificar días de la semana
dia_max_viajes = viajes_por_dia.loc[viajes_por_dia['total_viajes'].idxmax()]
dia_min_viajes = viajes_por_dia.loc[viajes_por_dia['total_viajes'].idxmin()]

print("\n### INSIGHTS DE PATRONES DE USO ###")
print(f"Día de mayor demanda: {dia_max_viajes['nombre_dia']} ({dia_max_viajes['total_viajes']:,} viajes)")
print(f"Día de menor demanda: {dia_min_viajes['nombre_dia']} ({dia_min_viajes['total_viajes']:,} viajes)")
print(f"Variación pico-valle: {(dia_max_viajes['total_viajes'] / dia_min_viajes['total_viajes'] - 1) * 100:.1f}%")

In [ ]:
# Visualización: Índice de Fidelización
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Viajes por día de la semana
colors_dias = ['#3498DB' if i < 5 else ['#E74C3C', '#9B59B6'][i-5] 
              for i in range(len(viajes_por_dia))]
bars = axes[0].bar(viajes_por_dia['nombre_dia'], viajes_por_dia['total_viajes'], 
                   color=colors_dias, alpha=0.7, edgecolor='black')
axes[0].axhline(y=viajes_promedio_dia, color='red', linestyle='--', 
               label=f'Promedio: {viajes_promedio_dia:,.0f}')
axes[0].set_title('Distribución de Viajes por Día de la Semana', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Día de la Semana')
axes[0].set_ylabel('Cantidad de Viajes')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=15)

# Gráfico 2: Indicadores de Fidelización
indicadores = ['Uso Consistente', 'Cobertura Horaria']
valores = [indice_uso_consistente, indice_cobertura_horaria]
colors_indicadores = ['#2ECC71', '#3498DB']

bars2 = axes[1].bar(indicadores, valores, color=colors_indicadores, 
                    alpha=0.7, edgecolor='black')
axes[1].set_ylim([0, 100])
axes[1].set_title('Indicadores de Fidelización y Uso Estable', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Índice (%)')
axes[1].grid(True, alpha=0.3, axis='y')

# Añadir etiquetas
for bar, valor in zip(bars2, valores):
    axes[1].text(bar.get_x() + bar.get_width()/2., valor,
                f'{valor:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Añadir zonas de referencia
axes[1].axhspan(60, 100, alpha=0.1, color='green', label='Alta Fidelización')
axes[1].axhspan(40, 60, alpha=0.1, color='yellow', label='Fidelización Media')
axes[1].axhspan(0, 40, alpha=0.1, color='red', label='Fidelización Baja')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

print("\n### CONCLUSIÓN DE NEGOCIO ###")
if indice_uso_consistente > 70:
    print("✅ ALTA FIDELIZACIÓN: Base de usuarios estable, activar programas de referidos")
elif indice_uso_consistente > 50:
    print("✅ FIDELIZACIÓN SALUDABLE: Foco en aumentar frecuencia de uso existente")
else:
    print("⚠️ FIDELIZACIÓN BAÑA: Revisar experiencia de usuario y retention strategies")

print(f"\n### NOTA METODOLÓGICA ###")
print("Este KPI usa proxies porque el dataset no contiene user_id.")
print("Para medición precisa, se requiere tracking de usuarios únicos.")

---

## RESUMEN EJECUTIVO DE LOS 7 KPIS

### **Tabla Resumen de KPIs**

| KPI | Fórmula | Unidad | Frecuencia | Dueño |
|-----|---------|--------|-------------|-------|
| 1. Tasa de Crecimiento de Ingresos | (Ingresos Actual - Ingresos Anterior) / Ingresos Anterior × 100 | % | Mensual | VP de Crecimiento |
| 2. Ticket Promedio | Total Ingresos / Total Viajes | USD | Semanal | Director de Pricing |
| 3. Factor de Ocupación | Pasajeros Promedio / Capacidad Máxima (6) | % | Diaria | Ops. de Flota |
| 4. Índice de Estacionalidad | Viajes Período / Viajes Promedio Mensual × 100 | Índice | Mensual | Planificación |
| 5. Eficiencia por Hora | Ingresos por Hora / Ingresos por Hora Promedio × 100 | % | Diaria | Gerente de Turnos |
| 6. Retorno por Kilómetro | Total Ingresos / Total Kilómetros | USD/KM | Semanal | Director Financiero |
| 7. Índice de Fidelización | Usuarios Múltiples Viajes / Total Usuarios × 100 | % | Mensual | Marketing |

### **Alineación con Hipótesis del Negocio**

1. **H1: Expansión Acelerada** → Validada por **KPI 1** (Tasa de Crecimiento)
2. **H2: Patrones Estacionales** → Medido por **KPI 4** (Índice de Estacionalidad)
3. **H3: Optimización por Horarios** → Evaluado por **KPI 5** (Eficiencia por Hora)
4. **H4: Rentabilidad por Segmentos** → Analizado por **KPI 2** (Ticket) y **KPI 6** (Retorno/KM)
5. **H5: Retención Impacta Ingresos** → Monitoreado por **KPI 7** (Fidelización) y **KPI 3** (Ocupación)

### **Próximos Pasos Recomendados**

1. **Automatización:** Configurar dashboards en Power BI con alertas automáticas
2. **Benchmarking:** Comparar KPIs contra industria y competidores
3. **Análisis Predictivo:** Modelar tendencias futuras basado en datos históricos
4. **Experimentación:** A/B testing para optimizar variables clave (pricing, incentivos)
5. **Segmentación Avanzada:** Incorporar datos demográficos y geográficos para análisis más granular